<a href="https://colab.research.google.com/github/7235SYXD/Real-Estate/blob/main/DSP_on_Real_Estate_(NB_3).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — INSTALL LIBRARIES                                      ║
# ╚══════════════════════════════════════════════════════════════════╝

import subprocess, sys
for lib in ["kagglehub","catboost","shap","optuna",
            "vaderSentiment","yake","textstat"]:
    subprocess.run([sys.executable,"-m","pip","install",lib,"-q"])
print("All libraries installed.")


All libraries installed.


In [2]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — GOOGLE DRIVE                                           ║
# ╚══════════════════════════════════════════════════════════════════╝

import os, shutil
from google.colab import drive, files

drive.mount('/content/drive')
SAVE_DIR = "/content/drive/MyDrive/RealEstate_TXNY"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")

Mounted at /content/drive
Save directory: /content/drive/MyDrive/RealEstate_TXNY


In [3]:
# ╔════════════════════════════════════════════════════╗
# ║  CELL 3 — IMPORT ALL LIBRARIES                     ║
# ╚════════════════════════════════════════════════════╝

import os
import re
import warnings
from pathlib import Path
import gc # Import the garbage collection module

import kagglehub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import math

# Sklearn
from sklearn.model_selection import train_test_split, KFold, cross_val_predict, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.inspection import permutation_importance
from sklearn.model_selection import cross_val_score
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, f1_score, classification_report,
    accuracy_score,
)

# Gradient boosting
!pip install catboost
from catboost import CatBoostRegressor, CatBoostClassifier
import xgboost as xgb

# Deep learning
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers as KL, Input as KInput, Model as KModel

# Hyperparameter tuning
!pip install optuna
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Interpretability
import shap

# NLP
!pip install vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
!pip install yake
import yake
!pip install textstat
import textstat

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", "{:,.4f}".format)

# Set random seeds for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("=" * 60)
print("All libraries imported successfully.")
print(f"  pandas     : {pd.__version__}")
print(f"  numpy      : {np.__version__}")
print(f"  tensorflow : {tf.__version__}")
print(f"  sklearn    : OK")
print("=" * 60)

All libraries imported successfully.
  pandas     : 2.2.2
  numpy      : 2.0.2
  tensorflow : 2.20.0
  sklearn    : OK


# Notebook 3 — Restore Checkpoint from Notebook 2

Loading the complete python session saved at the end of notebook 2. This
acts as a **connection point** between notebook 2 and 3.

## What Gets Restored

| Category | Examples |
|---|---|
| DataFrames | `df_combined` (149,999 × 51), `df_txny`, `X_train`, `X_val`, `X_test` |
| Numpy arrays | `Xtr_rf`, `Xva_rf`, `Xts_rf`, `Xtr_sc`, `Xva_sc`, `Xts_sc`, `Xtr_raw` |
| Targets | `y_reg_train`, `y_reg_val`, `y_reg_test`, `y_clf_train`, `y_clf_test` |
| Tuned models | `rf_tuned`, `cb_tuned` (from Cells 20, 21) |
| Best params | `rf_bp`, `cb_bp` (hyperparameter dicts for OOF consistency) |
| Results so far | `results_baseline` (4 entries), `results_tuned` (2 entries: RF + CB) |
| Constants | `SEED`, `final_features`, `NLP_COLS`, `BED_COL`, `BATH_COL` |
| Functions | `norm_y`, `denorm_y`, `evaluate_reg`, `add_leak_free_encoding` |
| Keras models | `mlp_base_model` (baseline MLP from Cell 19A) |





In [4]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  NOTEBOOK 3: SETUP — restores state from Notebook 2              ║
# ╚══════════════════════════════════════════════════════════════════╝
"""
Loads 338 variables from the dill checkpoint saved by Notebook 2.

Restore mechanism:
    1. pip install dill (fast, already cached)
    2. Open checkpoint_2_to_3.pkl → dill.load() → globals().update()
       Makes all 336 Notebook 2 variables available in this session.
    3. Open checkpoint_2_to_3_keras.json → for each entry,
       keras.models.load_model(path) → globals()[name] = model
       Restores mlp_base_model (the baseline 2-layer MLP from Cell 19A).

Confirmed restoration (from actual output):
    Variables restored    : 338
    Keras models restored : ['mlp_base_model']
    df_combined shape     : (119999, 51)

Critical variables confirmed available after restore:
    df_combined (119999, 51)  : full feature-engineered TX+NY corpus
    X_train / X_val / X_test  : leak-free feature metrices
    Xtr_rf, Xva_rf, Xts_rf    : imputed arrays for RF/CatBoost/HGB
    Xtr_sc, Xva_sc, Xts_sc    : scaled arrays for MLP
    Xtr_raw                   : raw arrays for HGB (native NaN support)
    y_reg_train/val/test      : log_price regression targets
    rf_tuned, cb_tuned        : tuned models from Cells 20, 21
    rf_bp, cb_bp              : best hyperparameter dicts
    results_baseline          : 4 baseline model metric dicts
    results_tuned             : 2 tuned model metric dicts (RF + CatBoost)
    norm_y, denorm_y          : MLP target normalisation functions
    n_feat                    : number of features (for MLP input shape)
"""

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "dill", "-q"])
import dill, json

CKPT_DIR  = f"{SAVE_DIR}/checkpoints"
CKPT_NAME = "checkpoint_2_to_3"

with open(f"{CKPT_DIR}/{CKPT_NAME}.pkl", "rb") as f:
    _state = dill.load(f)
globals().update(_state)

with open(f"{CKPT_DIR}/{CKPT_NAME}_keras.json") as f:
    _keras_paths = json.load(f)
for _name, _path in _keras_paths.items():
    globals()[_name] = keras.models.load_model(_path)

print("=" * 65)
print(f"CHECKPOINT RESTORED <- {CKPT_DIR}/{CKPT_NAME}.pkl")
print("=" * 65)
print(f"  Variables restored    : {len(_state)}")
print(f"  Keras models restored : {list(_keras_paths.keys()) or 'none'}")
if "df_combined" in globals():
    print(f"  df_combined shape      : {df_combined.shape}")
print(f"\nNotebook 2 state loaded successfully.")

CHECKPOINT RESTORED <- /content/drive/MyDrive/RealEstate_TXNY/checkpoints/checkpoint_2_to_3.pkl
  Variables restored    : 338
  Keras models restored : ['mlp_base_model']
  df_combined shape      : (119979, 51)

Notebook 2 state loaded successfully.


# CELL 22: MLP FINE-TUNING (RESIDUAL ARCHITECTURE)

Using Optuna with 30 trials fine tune a **residual block MLP**.
This residual architecture is considered as a major improvement over the
flat 2-layer baseline.

## Why Residual Blocks?

| Problem with flat MLP | Residual block solution |
|---|---|
| Vanishing gradients in deeper networks | Skip connections bypass dead layers
| No gradient flow between blocks | `Add()` layer ensures gradient reaches all weights |
| Single representation bottleneck | Two independent blocks learn complementary patterns

## Key Improvements vs Baseline MLP

| Aspect | Baseline (Cell 19A) | Tuned (Cell 22) |
|---|---|---|
| Architecture | 2-layer flat | Dual residual blocks |
| Loss | MSE | Huber (robust to price outliers) |
| LR schedule | Fixed Adam | Cosine decay (lr_max → 1e-6) |
| Target | Raw log_price | Normalised (zero mean, unit std) |
| Gradient clipping | None | clipnorm=1.0 |
| Regularisation | Dropout only | Dropout + L2 weight decay |




In [5]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 22 — OPTUNA: FINE-TUNE MLP (RESIDUAL ARCHITECTURE)         ║
# ╚══════════════════════════════════════════════════════════════════╝
"""
Using the Optuna TPE sampler, fine-tune a dual residual block MLP

Architecture — build_residual_mlp(u1, u2, dr, l2, n_input):
    Block 1 (same-dimension residual):
        x       = Dense(u1, l2_reg) → BN → ReLU → Dropout(dr)
        x2      = Dense(u1, l2_reg) → BN → ReLU → Dropout(dr×0.7)
        output  = Add([x, x2]) ← skip connection (same dimension)

    Block 2 (projected residual u1 → u2):
        x3      = Dense(u2, l2_reg) → BN → ReLU → Dropout(dr×0.5)
        x_proj  = Dense(u2, no bias) ← projection to match dimensions
        output  = Add([x3, x_proj]) ← projected residual connection

    Output: Dense(1) linear activation (regression)

    Args:
        u1      (int)   : units in Block 1 (first dense layer)
        u2      (int)   : units in Block 2 (second dense layer)
        dr      (float) : base dropout rate
        l2      (float) : L2 weight regularisation coefficient
        n_input (int)   : number of input features (from n_feat)
    Returns:
        Compiled Keras functional Model.

cosine_lr(epoch, total, lr_max, lr_min=1e-6):
    Cosine annealing learning rate schedule.
    lr = lr_min + 0.5 × (lr_max - lr_min) × (1 + cos(π × epoch / total))
    Starts with lr_max, decays smoothly to the lr_min=1e-6 over a total epochs.
    Applied through keras.callbacks.LearningRateScheduler.

mlp_objective(trial) — Optuna objective:
    Hyperparameter search space:
        u1    : {128, 256, 512}         — Block 1 units
        u2    : {64, 128, 256}          — Block 2 units
        dr    : [0.1, 0.4]              — dropout rate
        lr    : [5e-4, 3e-3] log-scale  — peak learning rate
        bs    : {256, 512, 1024}        — batch size
        l2    : [1e-5, 1e-3] log-scale  — L2 regularisation
        delta : [0.5, 2.0]              — Huber loss delta parameter

    Training configuration:
        Loss      : Huber(delta=delta)        — robust to price outliers vs MSE
        Optimizer : Adam(lr, clipnorm=1.0)    — gradient clipping prevents explosions
        Target    : yr_tr_sc (normalised)     — zero mean, unit variance
        Callbacks : EarlyStopping(patience=15), ReduceLROnPlateau(factor=0.5, patience=5)

    Returns: r2_score(y_reg_val, denorm_y(predictions))

Final model (mlp_tuned):
    Trained with the N_FINAL=200 epochs using a best Optuna params.
    Same callbacks as objective + cosine LR scheduler.

Output variables:
    mlp_tuned   (keras.Model)  : final fitted residual MLP
    bp_mlp      (dict)         : best hyperparameters from Optuna
    N_MLP_EPOCHS (int)         : 120 — epochs per Optuna trial
    N_FINAL      (int)         : 200 — epochs for final mlp_tuned fit
    results_tuned[2]           : metrics dict for MLP tuned
"""

print("=" * 65)
print("CELL 22 — OPTUNA: MLP RESIDUAL ARCHITECTURE (30 trials)")
print("=" * 65)

def build_residual_mlp(u1, u2, dr, l2, n_input):
    """
    Dual residual block MLP.
    Block 1: Dense(u1) → BN → ReLU → Dropout → Dense(u1) + skip
    Block 2: Dense(u2) → BN → ReLU → Dropout + projected skip
    Output: Dense(1) linear
    """
    reg = keras.regularizers.l2(l2)
    inp = KInput(shape=(n_input,))

    # Block 1 (same-dimension residual)
    x  = KL.Dense(u1, kernel_regularizer=reg)(inp)
    x  = KL.BatchNormalization()(x)
    x  = KL.Activation("relu")(x)
    x  = KL.Dropout(dr)(x)
    x2 = KL.Dense(u1, kernel_regularizer=reg)(x)
    x2 = KL.BatchNormalization()(x2)
    x2 = KL.Activation("relu")(x2)
    x2 = KL.Dropout(dr * 0.7)(x2)
    x  = KL.Add()([x, x2])

    # Block 2 (projected residual: u1 → u2)
    x3 = KL.Dense(u2, kernel_regularizer=reg)(x)
    x3 = KL.BatchNormalization()(x3)
    x3 = KL.Activation("relu")(x3)
    x3 = KL.Dropout(dr * 0.5)(x3)
    xp = KL.Dense(u2, use_bias=False)(x)   # projection
    x  = KL.Add()([x3, xp])

    out   = KL.Dense(1)(x)
    model = KModel(inputs=inp, outputs=out)
    return model

def cosine_lr(epoch, total, lr_max, lr_min=1e-6):
    # Cosine decay from lr_max to lr_min over total epochs.
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + math.cos(math.pi * epoch / total))

N_MLP_EPOCHS = 120

def mlp_objective(trial):
    # Maximise validation R² with improved MLP architecture.
    u1    = trial.suggest_categorical  ("u1", [128, 256, 512])
    u2    = trial.suggest_categorical  ("u2", [64,  128, 256])
    dr    = trial.suggest_float        ("dr", 0.1,  0.4)
    lr    = trial.suggest_float        ("lr", 5e-4, 3e-3, log=True)
    bs    = trial.suggest_categorical  ("bs", [256,  512, 1024])
    l2    = trial.suggest_float        ("l2", 1e-5, 1e-3, log=True)
    delta = trial.suggest_float        ("delta", 0.5,  2.0)

    m = build_residual_mlp(u1, u2, dr, l2, n_feat)
    m.compile(optimizer=keras.optimizers.Adam(lr, clipnorm=1.0),
               loss=keras.losses.Huber(delta=delta))
    lr_cb = keras.callbacks.LearningRateScheduler(
        lambda ep: cosine_lr(ep, N_MLP_EPOCHS, lr_max=lr))
    m.fit(Xtr_sc, yr_tr_sc,
          validation_data=(Xva_sc, yr_va_sc),
          epochs=N_MLP_EPOCHS, batch_size=bs, verbose=0,
          callbacks=[
              keras.callbacks.EarlyStopping(
                  patience=15, min_delta=1e-4,
                  restore_best_weights=True),
              keras.callbacks.ReduceLROnPlateau(
                  monitor="val_loss", factor=0.5, patience=5,
                  min_lr=1e-6, verbose=0),
              lr_cb,
          ])
    preds = denorm_y(m.predict(Xva_sc, verbose=0).flatten())
    return r2_score(y_reg_val.values, preds)

mlp_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED))
mlp_study.optimize(mlp_objective, n_trials=30, show_progress_bar=True)
bp_mlp = mlp_study.best_params
print(f"\nBest MLP params : {bp_mlp}")

# Train final MLP with best params
N_FINAL = 200
mlp_tuned = build_residual_mlp(
    bp_mlp["u1"], bp_mlp["u2"], bp_mlp["dr"], bp_mlp["l2"], n_feat)
mlp_tuned.compile(
    optimizer=keras.optimizers.Adam(bp_mlp["lr"], clipnorm=1.0),
    loss=keras.losses.Huber(delta=bp_mlp["delta"]))
lr_final_cb = keras.callbacks.LearningRateScheduler(
    lambda ep: cosine_lr(ep, N_FINAL, lr_max=bp_mlp["lr"]))
mlp_tuned.fit(Xtr_sc, yr_tr_sc,
               validation_data=(Xva_sc, yr_va_sc),
               epochs=N_FINAL, batch_size=bp_mlp["bs"], verbose=0,
               callbacks=[
                   keras.callbacks.EarlyStopping(
                       patience=20, min_delta=1e-4,
                       restore_best_weights=True),
                   keras.callbacks.ReduceLROnPlateau(
                       monitor="val_loss", factor=0.5, patience=5,
                       min_lr=1e-6, verbose=0),
                   lr_final_cb,
               ])

mlp_tuned_preds = denorm_y(mlp_tuned.predict(Xva_sc, verbose=0).flatten())
mlp_bl_r2  = results_baseline[2]["R2"]
res_mlp_t  = evaluate_reg(y_reg_val, mlp_tuned_preds, "MLP (residual, tuned)", "val")
results_tuned.append(res_mlp_t)
print(f"\nMLP improvement: {mlp_bl_r2:.4f} → {res_mlp_t['R2']:.4f}  "
      f"({res_mlp_t['R2'] - mlp_bl_r2:+.4f})")

CELL 22 — OPTUNA: MLP RESIDUAL ARCHITECTURE (30 trials)


  0%|          | 0/30 [00:00<?, ?it/s]


Best MLP params : {'u1': 256, 'u2': 64, 'dr': 0.10127897615858379, 'lr': 0.0024194395166600626, 'bs': 512, 'l2': 0.0005450721698389965, 'delta': 1.8798183527231516}
  MLP (residual, tuned)                      [val] R²=0.7517  RMSE=0.4573  MAE=0.1965  Real-MAE=$68,507

MLP improvement: 0.7329 → 0.7517  (+0.0188)


# Cell 23 — Optuna Fine-Tuning: HistGradientBoosting (20 Trials)

Using Optuna trails fine-tune sklearn's **HistGradientBoostingRegressor**

## Why HGB in This Ensemble?

| Property | CatBoost | HGB |
|---|---|---|
| NaN handling | Requires imputation | Native NaN support |
| Speed (120K rows) | Moderate | Very fast (histogram binning) |
| Inductive bias | Symmetric trees, ordered boosting | Standard CART trees |
| Ensemble diversity | — | Different bias → better stack |

HGB uses `Xtr_raw` instead of `Xtr_rf` because HGB handles NaN natively.
That means zip_numeric which contains 30% NaN and latitude/longitude which contains 82% NaN are passed directly without any imputation - So we can say that HGB learns to split around NaN automatically.

## Search Space

| Parameter | Range | Why |
|---|---|---|
| `learning_rate` | 0.01–0.3 (log) | Log-uniform for better low-value coverage |
| `max_iter` | 200–1000 (step 100) | Total boosting rounds |
| `max_depth` | 3–10 | Tree complexity |
| `min_samples_leaf` | 10–100 | Leaf regularisation |
| `l2_regularization` | 0.0–10.0 | Weight shrinkage |
| `max_bins` | {63, 127, 255} | Histogram resolution |




In [6]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CELL 23 — OPTUNA: FINE-TUNE HISTGRADIENTBOOSTING                ║
# ╚══════════════════════════════════════════════════════════════════╝
"""
Fine-tunes HistGradientBoostingRegressor using Optuna TPE.

Key design:
 HGB uses a raw numpy arrays i.e, Xtr_raw and not the imputed one which is Xtr_rf.
 The reason behind this is the missing values which is handled by HGB natively by
 using a dedicated NaN bin in its histogram. Passing these imputed values would fill NaN
 cells, but it reduces the HGB's performance on features with high null rates.

hgb_objective(trial) — Optuna objective:
    Hyperparameter search space:
        learning_rate    : [0.01, 0.3]  log-uniform — boosting step size
        max_iter         : [200, 1000]  step=100    — max boosting rounds
        max_depth        : [3, 10]                  — max tree depth per round
        min_samples_leaf : [10, 100]                — min samples per leaf
        l2_regularization: [0.0, 10.0]              — L2 shrinkage
        max_bins         : {63, 127, 255}           — histogram bin count
                          More bins = finer splits, but more memory/time.

    Early stopping (inside model, not Optuna):
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=15
        HGB allocates 10% of the training data for internal early stopping.
        This prevents the model from overfitting within each trial.

    Returns: r2_score(y_reg_val, m.predict(Xva_raw))

Final model (hgb_tuned):
    Same params as the best trial, but early_stopping=False for final fit.

Output variables:
    hgb_tuned   (HistGradientBoostingRegressor) : fitted with best params
    hgb_bp      (dict) : best params. Keys: lr, max_iter, depth, min_leaf, l2, bins
    results_tuned[3]   : metrics dict for HGB tuned (appended to list)
"""

print("=" * 65)
print("CELL 23 — OPTUNA: HISTGRADIENTBOOSTING (20 trials)")
print("=" * 65)

def hgb_objective(trial):
    #Maximise validation R². HGB uses raw arrays.
    params = {
        "learning_rate":     trial.suggest_float("lr",          0.01, 0.3, log=True),
        "max_iter":          trial.suggest_int("max_iter",      200,  1000, step=100),
        "max_depth":         trial.suggest_int("depth",         3,    10),
        "min_samples_leaf":  trial.suggest_int("min_leaf",      10,   100),
        "l2_regularization": trial.suggest_float("l2",          0.0,  10.0),
        "max_bins":          trial.suggest_categorical("bins",   [63, 127, 255]),
        "early_stopping":    True,
        "validation_fraction": 0.1,
        "n_iter_no_change":  15,
        "random_state": SEED,
    }
    m = HistGradientBoostingRegressor(**params)
    m.fit(Xtr_raw, y_reg_train)
    return r2_score(y_reg_val, m.predict(Xva_raw))

hgb_study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED))
hgb_study.optimize(hgb_objective, n_trials=20, show_progress_bar=True)
hgb_bp = hgb_study.best_params
print(f"\nBest HGB params : {hgb_bp}")

hgb_tuned = HistGradientBoostingRegressor(
    learning_rate    = hgb_bp["lr"],
    max_iter         = hgb_bp["max_iter"],
    max_depth        = hgb_bp["depth"],
    min_samples_leaf = hgb_bp["min_leaf"],
    l2_regularization= hgb_bp["l2"],
    max_bins         = hgb_bp["bins"],
    early_stopping   = True,
    validation_fraction=0.1,
    n_iter_no_change = 15,
    random_state=SEED)
hgb_tuned.fit(Xtr_raw, y_reg_train)

hgb_bl_r2  = results_baseline[3]["R2"]
res_hgb_t  = evaluate_reg(y_reg_val, hgb_tuned.predict(Xva_raw), "HGB (tuned)", "val")
results_tuned.append(res_hgb_t)
print(f"\nHGB improvement: {hgb_bl_r2:.4f} → {res_hgb_t['R2']:.4f}  "
      f"({res_hgb_t['R2'] - hgb_bl_r2:+.4f})")


CELL 23 — OPTUNA: HISTGRADIENTBOOSTING (20 trials)


  0%|          | 0/20 [00:00<?, ?it/s]


Best HGB params : {'lr': 0.27442950854269327, 'max_iter': 200, 'depth': 7, 'min_leaf': 10, 'l2': 0.08852353465206728, 'bins': 255}
  HGB (tuned)                                [val] R²=0.7682  RMSE=0.4418  MAE=0.1975  Real-MAE=$68,575

HGB improvement: 0.7604 → 0.7682  (+0.0078)


# Notebook 3 — Final Cell: Save Checkpoint for Notebook 4

Saves all tuned models, and session state to Google Drive.



In [7]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  CHECKPOINT — SAVE STATE FOR Notebook 4                          ║
# ╚══════════════════════════════════════════════════════════════════╝

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "dill", "-q"])
import dill, types, json, os

CKPT_DIR  = f"{SAVE_DIR}/checkpoints"
os.makedirs(CKPT_DIR, exist_ok=True)
CKPT_NAME = "checkpoint_3_to_4"

_keras_models = {}
_state        = {}
_skipped      = []
_reserved     = {"_keras_models", "_state", "_skipped", "_reserved",
                  "_name", "_val", "_path", "CKPT_DIR", "CKPT_NAME"}

for _name, _val in list(globals().items()):
    if _name.startswith("_") or _name in _reserved:
        continue
    try:
        if isinstance(_val, types.ModuleType):
            continue
        if type(_val).__module__.startswith("matplotlib"):
            continue
        if isinstance(_val, tf.keras.Model):
            _path = f"{CKPT_DIR}/{CKPT_NAME}__{_name}.keras"
            _val.save(_path)
            _keras_models[_name] = _path
            continue
        dill.dumps(_val)
        _state[_name] = _val
    except Exception:
        _skipped.append(_name)

with open(f"{CKPT_DIR}/{CKPT_NAME}.pkl", "wb") as f:
    dill.dump(_state, f)
with open(f"{CKPT_DIR}/{CKPT_NAME}_keras.json", "w") as f:
    json.dump(_keras_models, f)

print("=" * 65)
print(f"CHECKPOINT SAVED -> {CKPT_DIR}/{CKPT_NAME}.pkl")
print("=" * 65)
print(f"  Variables saved    : {len(_state)}")
print(f"  Keras models saved : {list(_keras_models.keys()) or 'none'}")
if _skipped:
    print(f"  Skipped (not needed downstream / not picklable): {_skipped}")
print("from its first cell — it will restore this state automatically.")

CHECKPOINT SAVED -> /content/drive/MyDrive/RealEstate_TXNY/checkpoints/checkpoint_3_to_4.pkl
  Variables saved    : 355
  Keras models saved : ['mlp_base_model', 'mlp_tuned']
  Skipped (not needed downstream / not picklable): ['get_ipython', 'exit', 'quit']
from its first cell — it will restore this state automatically.
